# Phase 1: Retrieval Basics

This notebook covers the Phase-1 objectives:
- Implement and compare **TF-IDF**, **BM25+**, and **Embedding-based** retrieval.
- Evaluate with standard IR metrics: **Precision@K**, **Recall@K**, **MRR@K**, **MAP@K**.

The dense model is optional at runtime because it is slower.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path('../..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_data, build_qrels_lookup
from src.preprocess import create_content_column
from src.models import run_tfidf_search, run_bm25_search, run_dense_search
from src.evaluation import evaluate_retrieval


In [ ]:
DATA_DIR = PROJECT_ROOT / 'data'
EVAL_K = 10

# Subset controls to keep notebook practical.
DOC_LIMIT = 10000
QUERY_LIMIT = 200

RUN_DENSE = True
DENSE_QUERY_LIMIT = 50

docs_df, train_queries_df, _, qrels_raw = load_data(DATA_DIR)
qrels = build_qrels_lookup(qrels_raw)

docs_df = create_content_column(docs_df, ['title', 'text', 'tags'])
train_queries_df = create_content_column(train_queries_df, ['title', 'text'])

docs_eval = docs_df.head(DOC_LIMIT).copy()
queries_eval = train_queries_df.head(QUERY_LIMIT).copy()

print('docs_eval:', docs_eval.shape)
print('queries_eval:', queries_eval.shape)
print('EVAL_K:', EVAL_K)


In [ ]:
def evaluate_model(name, retrieval_results, qrels_lookup, k):
    metrics = evaluate_retrieval(retrieval_results, qrels_lookup, k=k)
    row = {'model': name}
    row.update(metrics)
    row['queries_evaluated'] = len(retrieval_results)
    return row


results_table = []

# TF-IDF
tfidf_results = run_tfidf_search(docs_eval, queries_eval, top_k=EVAL_K)
qids_eval = queries_eval['id'].astype(str).tolist()
qrels_subset = {qid: qrels.get(qid, []) for qid in qids_eval}
results_table.append(evaluate_model('TF-IDF', tfidf_results, qrels_subset, EVAL_K))

# BM25+
bm25_results = run_bm25_search(docs_eval, queries_eval, top_k=EVAL_K)
results_table.append(evaluate_model('BM25+', bm25_results, qrels_subset, EVAL_K))

# Dense (embedding-based)
if RUN_DENSE:
    dense_queries = queries_eval.head(DENSE_QUERY_LIMIT).copy()
    dense_results = run_dense_search(docs_eval, dense_queries, top_k=EVAL_K)
    dense_qids = dense_queries['id'].astype(str).tolist()
    dense_qrels_subset = {qid: qrels.get(qid, []) for qid in dense_qids}
    results_table.append(evaluate_model('Dense Embedding', dense_results, dense_qrels_subset, EVAL_K))
else:
    results_table.append({
        'model': 'Dense Embedding (skipped)',
        'avg_recall': float('nan'),
        'avg_precision': float('nan'),
        'mrr': float('nan'),
        'map': float('nan'),
        'queries_evaluated': 0,
    })

comparison_df = pd.DataFrame(results_table)
comparison_df


In [ ]:
        # Optional: compare the top documents for one query across models.
        example_idx = 0
        query_id = str(queries_eval.iloc[example_idx]['id'])
        query_text = queries_eval.iloc[example_idx]['content']

        print('query_id:', query_id)
        print('query_text:', query_text[:300], '...')

        print('
TF-IDF top docs:')
        print(tfidf_results[example_idx]['relevant_docs'][:5])

        print('
BM25+ top docs:')
        print(bm25_results[example_idx]['relevant_docs'][:5])

        if RUN_DENSE:
            print('
Dense top docs:')
            print(dense_results[0]['relevant_docs'][:5])


## Interpretation

- **TF-IDF** and **BM25+** are lexical baselines.
- **Dense embedding retrieval** captures semantic similarity beyond exact token overlap.
- The metric table above is the Phase-1 baseline comparison.
